# Continuous ASL Foundation Model — Kaggle TPU v5e-1 Experimentation Template

This notebook provides the **official high-throughput runtime template** for training and experimenting with the Continuous ASL Foundation Model on **Google Cloud TPU v5e-1 (1 chip / 16 GB HBM)** and **TPU v5e-8 (4 chips / 64 GB HBM)**.

### Architectural Configurations (Enforced Invariants):
- **`--d-model 512`** (4 systolic tiles of 128)
- **`--max-len 384`** (3 systolic tiles of 128)
- **`--english-max-len 128`** (1 systolic tile of 128)
- **`--chicago-max-len 128`** (1 systolic tile of 128)
- **`--batch-size 2048`** (16 systolic tiles of 128)

---

### How to Assign this TPU to Antigravity:
1. In the Kaggle top menu, go to **Run** -> **Kaggle Jupyter Server**.
2. Copy the provided URL (e.g. `https://kkb-production.jupyter-proxy.kaggle.net?token=...`).
3. Paste the URL into Antigravity chat, or run locally:
   ```powershell
   python scratch/assign_tpu.py --url "<YOUR_COPIED_URL>"
   ```

## Cell 1: Official Kaggle TPU v5e Kernel Tuning & Memory Setup
Enables Transparent Hugepages (THP) and sets high-throughput memory allocator headroom (`0.95`).

In [ ]:
import os
import sys

# 1. Enable Transparent Hugepages (THP) for TPU v5e-1 runtime
!sudo sh -c 'echo always > /sys/kernel/mm/transparent_hugepage/enabled' 2>/dev/null || true
!sudo sh -c 'echo always > /sys/kernel/mm/transparent_hugepage/defrag' 2>/dev/null || true

# 2. IPC pipe and memory mapping limits
!sudo sysctl -w vm.max_map_count=1048576 2>/dev/null || true
!sudo sh -c 'echo 268435456 > /proc/sys/fs/pipe-max-size' 2>/dev/null || true

# 3. Enforce 95% HBM allocation headroom (unlocks 14.5 GiB / 15.25 GiB)
os.environ["KERAS_BACKEND"] = "jax"
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["PJRT_ALLOCATOR_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_CLIENT_MEM_FRACTION"] = "0.95"

# 4. Fast LibTPU Matrix Multiply & Collective Fusion Flags
os.environ["LIBTPU_INIT_ARGS"] = (
    "--xla_tpu_enable_flash_attention=true "
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_enable_async_collective_fusion=true "
    "--xla_tpu_enable_async_collective_fusion_multiple_steps=true "
    "--xla_tpu_rwb_fusion=true"
)

print("[+] TPU v5e-1 Kernel Tuning Applied Successfully!")

## Cell 2: Hardware Verification (JAX & PyTorch/XLA Diagnostics)
Verifies hardware attachment and systolic tile compatibility.

In [ ]:
import jax
print("[+] JAX Platform:", jax.default_backend())
print("[+] JAX Devices:", jax.devices())
print(f"[+] Device Count: {jax.device_count()}, Local: {jax.local_device_count()}")

try:
    import torch
    import torch_xla.core.xla_model as xm
    dev = xm.xla_device()
    print(f"[+] PyTorch/XLA Attached Device: {dev}")
except Exception as e:
    print(f"[!] PyTorch/XLA Probe: {e}")

!cat /proc/cpuinfo | grep 'model name' | head -n 1
!free -h

## Cell 3: Clone / Sync Latest Codebase & Datasets

In [ ]:
import os

REPO_DIR = "/kaggle/working/netmaui-singlanguage"
if not os.path.exists(REPO_DIR):
    print("[*] Cloning repository...")
    !git clone https://github.com/Codering2012/netmaui-singlanguage.git {REPO_DIR}
else:
    print("[*] Pulling latest updates...")
    !cd {REPO_DIR} && git pull

# Change directory to the repository root
%cd {REPO_DIR}

# Verify critical files
assert os.path.exists("train.py"), "train.py not found!"
assert os.path.exists("train_keras/train_tpu_keras.py"), "train_tpu_keras.py not found!"
print("[+] Codebase verified successfully!")

## Cell 4: Launch High-Speed Keras 3 + JAX Training on TPU v5e-1
Runs the hardware-fused scaled dot-product attention pipeline with zero-copy DLPack prefetching.

In [ ]:
# Execute Keras 3 (JAX Backend) Continuous ASL Training
!python -u train_keras/train_tpu_keras.py \
  --data-dir /kaggle/input/datasets/tranquocbao2012/frakenstein-asl-final-version/asl_dataset/asl_preprocessed_phase1 \
  --kdwd-dir /kaggle/input/datasets/kenshoresearch/kensho-derived-wikimedia-data \
  --aslg-csv /kaggle/input/datasets/thedevastator/unlock-the-power-of-english-asl-with-aslg-pc12-c/train.csv \
  --phase1-checkpoint /kaggle/input/models/muddragonmike/pretrain-bidirectional-asl-english/pytorch/default/1/asl_llm_200 \
  --save-dir /kaggle/working/checkpoints \
  --batch-size 2048 \
  --lr 5e-4 \
  --epochs 100 \
  --max-len 384 \
  --english-max-len 128 \
  --chicago-max-len 128 \
  --d-model 512 \
  --nhead 4 \
  --kv-heads 2 \
  --num-layers 4 \
  --precision mixed_bfloat16 \
  --log-freq 25 \
  --checkpoint-freq 20 \
  --keep-last-k 5

## Cell 5: Launch PyTorch/XLA Training on TPU v5e-1
Runs the monolithic PyTorch/XLA pipeline with resolved HBM reservation headroom.

In [ ]:
!PJRT_DEVICE=TPU python -u train.py \
  --data-dir /kaggle/input/datasets/tranquocbao2012/frakenstein-asl-final-version/asl_dataset/asl_preprocessed_phase1 \
  --kdwd-dir /kaggle/input/datasets/kenshoresearch/kensho-derived-wikimedia-data \
  --aslg-csv /kaggle/input/datasets/thedevastator/unlock-the-power-of-english-asl-with-aslg-pc12-c/train.csv \
  --phase1-checkpoint /kaggle/input/models/muddragonmike/pretrain-bidirectional-asl-english/pytorch/default/1/asl_llm_200 \
  --save-dir /kaggle/working/checkpoints \
  --streamed-dataset \
  --tpu \
  --precision bfloat16 \
  --batch-size 2048 \
  --batches-per-execution 4 \
  --num-dataloader-workers 0 \
  --accum-steps 1 \
  --epochs 100 \
  --phase1-epochs 0 \
  --max-len 384 \
  --english-max-len 128 \
  --chicago-max-len 128 \
  --d-model 512 \
  --nhead 4 \
  --num-layers 4 \
  --log-freq 25 \
  --is-causal \
  --keep-last-k 5 \
  --checkpoint-freq 20 \
  --use-gpt2 \
  2>&1 | tee /kaggle/working/train.log

## Cell 6: Background Keep-Alive Heartbeat
Runs a passive heartbeat loop so the Kaggle TPU session does not timeout while Antigravity runs experiments.

In [ ]:
import time
print("[*] Session keep-alive active. Press Stop in Kaggle UI to terminate.")
for i in range(1, 1440):
    time.sleep(60)
    if i % 10 == 0:
        print(f"[Heartbeat] TPU v5e-1 session active: {i} minutes elapsed.", flush=True)